# Row 4: the matcher decides when to ask, the decider decides whether it is the moment

What this tests and why: `docs/RATIONALE.md` (section 4, row 4; decisions D13, D20). It reuses row 3's stored similarities and row 1's
decider, so you do **not** re-run either. Only the decider runs, and only at the moments the matcher fires (about 1,800 calls on val,
roughly 10 minutes). Run the cells in order from the repo root.

A response needs both: the matcher fired **and** the decider said Yes. So this can only remove responses relative to the matcher alone;
the question is whether it removes more false alarms than true events.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":      # run from the repo root
    os.chdir("..")
print(os.getcwd())
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Find the two caches

Row 3's similarities (matcher cache) and the decider cache. If either line prints nothing, run row 3 / row 1 first.

In [ ]:
import glob
MATCHER = sorted(glob.glob("results/cache/matcher/Ops-MM-embedding-v1-2B__*"))[-1]
print("matcher:", MATCHER)
print("decider:", sorted(glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_only__*")))

## 2. Smoke test: three questions

In [ ]:
!python scripts/run_gated_decider.py --samples data/processed/ovo_far_val.jsonl --matcher {MATCHER} --limit 3

## 3. Speed check

Median milliseconds per on-demand decider call (row 1 measured about 300).

In [ ]:
import glob, json, statistics
rows = [json.loads(l) for f in glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_only__*/ondemand/*.jsonl") for l in open(f) if l.strip()]
print(len(rows), "on-demand calls; median", round(statistics.median(r["ms"] for r in rows)), "ms")

## 4. All val questions (resumable; never repeats a decider call)

In [ ]:
!python scripts/run_gated_decider.py --samples data/processed/ovo_far_val.jsonl --matcher {MATCHER}

## 5. Score it

Rows are matcher thresholds and cooldowns; columns are how sure the decider must be (tau 0.00 means the matcher alone, nothing removed).
Compare the best cell with row 1 (about 30 at similar response counts) and row 3 (about 30).

In [ ]:
import glob
DECIDER = sorted(glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_only__*"))[-1]
!python scripts/replay_gated.py --matcher {MATCHER} --decider {DECIDER} --samples data/processed/ovo_far_val.jsonl

## 6. Push the results

```
git add results/cache && git commit -m "Row 4 on-demand decider answers and replay, val" && git pull --rebase origin abhi && git push origin abhi
```
Please also push `results/cache/matcher` and `results/cache/cues` if you have not: the small `.json` files are what let per-question and per-task breakdowns be computed.